In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata'] 
coleccion = db['AutoTec_scraping'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [5]:
import subprocess
subprocess.run(["pip", "install", "undetected-chromedriver"], check=True)
print("✅ Instalado correctamente.")

✅ Instalado correctamente.


In [30]:
#eliminar datos de mongo carpeta AutoTec_scraping
from pymongo import MongoClient

client = MongoClient("172.18.0.1", 27017)
db = client["proyecto_bigdata"]
coleccion = db["AutoTec_scraping"]

resultado = coleccion.delete_many({})
print(f"Registros eliminados: {resultado.deleted_count}")

Registros eliminados: 120


In [21]:
# --- PASO 0: LIMPIEZA ---
import os
import time
from pymongo import MongoClient
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "AutoTec"
USUARIO = "Belen A"
lista_autos = []
driver = None

# --- PASO 1: CONFIGURACION DEL NAVEGADOR ---
options = uc.ChromeOptions()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--headless=new")

try:
    driver = uc.Chrome(options=options, version_main=147)
    print("Navegador iniciado correctamente.")

    limite_paginas = 10
    URL_BASE = "https://callegari.cl/seminuevos/page/{}"

    for nivel_pagina in range(1, limite_paginas + 1):
        url_pagina = URL_BASE.format(nivel_pagina)
        print(f"--- Procesando Pagina {nivel_pagina} ---")

        driver.get(url_pagina)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(6)

        bloques = driver.find_elements(By.CSS_SELECTOR, "a.auto-block")
        print(f"  -> {len(bloques)} autos encontrados.")

        for bloque in bloques:
            try:
                url_auto = bloque.get_attribute("href")

                # Extraemos todo el texto y separamos por lineas
                lineas = [l.strip() for l in bloque.text.strip().split("\n") if l.strip()]

                # Detectar si hay badge en la primera linea
                badges = ["Auto Empresa", "Garantia Fabrica", "Garantía Fábrica", "Unico Dueno", "Único Dueño"]
                inicio = 1 if any(b.lower() in lineas[0].lower() for b in badges) else 0

                marca  = lineas[inicio + 0] if len(lineas) > inicio + 0 else "N/A"
                modelo = lineas[inicio + 1] if len(lineas) > inicio + 1 else "N/A"
                precio = lineas[inicio + 3] if len(lineas) > inicio + 3 else "0"

                if len(lineas) > inicio + 4:
                    partes      = [p.strip() for p in lineas[inicio + 4].split("|")]
                    anio        = partes[0] if len(partes) > 0 else "N/A"
                    kilometraje = partes[1] if len(partes) > 1 else "N/A"
                    combustible = partes[3] if len(partes) > 3 else "N/A"
                else:
                    anio = kilometraje = combustible = "N/A"

                lista_autos.append({
                    "marca":         marca,
                    "modelo":        modelo,
                    "anio":          anio,
                    "kilometraje":   kilometraje,
                    "combustible":   combustible,
                    "ciudad":        "N/A",
                    "url":           url_auto,
                    "precio":        precio,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo":         NOMBRE_GRUPO,
                    "usuario":       USUARIO
                })

            except Exception:
                continue

        print(f"  Acumulado total: {len(lista_autos)} autos.")
        time.sleep(2)

    print(f"Extraccion terminada: {len(lista_autos)} autos en total.")

except Exception as e:
    print(f"Error en Selenium: {e}")

finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    client = MongoClient("172.18.0.1", 27017, serverSelectionTimeoutMS=5000)
    db = client["proyecto_bigdata"]
    coleccion = db["AutoTec_scraping"]

    if lista_autos:
        for d in lista_autos:
            # Precio -> float
            v_limpio = str(d["precio"]).replace(".", "").replace(",", "").replace("$", "").strip()
            try:
                d["precio"] = float(v_limpio) if v_limpio else 0.0
            except ValueError:
                d["precio"] = 0.0

            # Kilometraje -> int
            km_limpio = str(d["kilometraje"]).replace(".", "").replace("Km", "").replace("km", "").strip()
            try:
                d["kilometraje"] = int(km_limpio) if km_limpio else 0
            except ValueError:
                d["kilometraje"] = 0

            # Anio -> int
            try:
                d["anio"] = int(d["anio"])
            except ValueError:
                d["anio"] = 0

        coleccion.insert_many(lista_autos)
        print(f"{len(lista_autos)} autos cargados en MongoDB correctamente.")
    else:
        print("No hay datos para guardar.")

except Exception as e:
    print(f"Error en MongoDB: {e}")

Limpieza de procesos y temporales completada.
Navegador iniciado correctamente.
--- Procesando Pagina 1 ---
  -> 12 autos encontrados.
  Acumulado total: 12 autos.
--- Procesando Pagina 2 ---
  -> 12 autos encontrados.
  Acumulado total: 24 autos.
--- Procesando Pagina 3 ---
  -> 12 autos encontrados.
  Acumulado total: 36 autos.
--- Procesando Pagina 4 ---
  -> 12 autos encontrados.
  Acumulado total: 48 autos.
--- Procesando Pagina 5 ---
  -> 12 autos encontrados.
  Acumulado total: 60 autos.
--- Procesando Pagina 6 ---
  -> 12 autos encontrados.
  Acumulado total: 72 autos.
--- Procesando Pagina 7 ---
  -> 12 autos encontrados.
  Acumulado total: 84 autos.
--- Procesando Pagina 8 ---
  -> 12 autos encontrados.
  Acumulado total: 96 autos.
--- Procesando Pagina 9 ---
  -> 12 autos encontrados.
  Acumulado total: 108 autos.
--- Procesando Pagina 10 ---
  -> 4 autos encontrados.
  Acumulado total: 112 autos.
Extraccion terminada: 112 autos en total.
112 autos cargados en MongoDB correc

In [24]:
# --- PASO 0: LIMPIEZA ---
import os
import time
from pymongo import MongoClient
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")

print("Limpieza completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "AutoTec"
USUARIO = "Belen A"

lista_autos = []
driver = None

# --- CONFIGURACION CHROME ---
options = uc.ChromeOptions()

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--headless=new")

try:

    driver = uc.Chrome(options=options, version_main=147)

    print("Navegador iniciado correctamente.")

    limite_paginas = 5

    URL_BASE = "https://seminuevosvalentini.cl/search/page/{}/"

    for pagina in range(1, limite_paginas + 1):

        url = URL_BASE.format(pagina)

        print(f"\n--- PROCESANDO PAGINA {pagina} ---")

        driver.get(url)

        time.sleep(5)

        driver.execute_script(
            "window.scrollTo(0, document.body.scrollHeight);"
        )

        time.sleep(3)

        # BLOQUES DE AUTOS
        bloques = driver.find_elements(
            By.CSS_SELECTOR,
            "div.vehica-car-card__content"
        )

        print(f"Autos encontrados: {len(bloques)}")

        for bloque in bloques:

            try:

                # NOMBRE + URL
                nombre_elemento = bloque.find_element(
                    By.CSS_SELECTOR,
                    "a.vehica-car-card__name"
                )

                nombre = nombre_elemento.text.strip()

                url_auto = nombre_elemento.get_attribute("href")

                # PRECIO
                try:
                    precio = bloque.find_element(
                        By.CSS_SELECTOR,
                        "div.vehica-car-card__price"
                    ).text.strip()
                except:
                    precio = 0

                # INFORMACION EXTRA
                infos = bloque.find_elements(
                    By.CSS_SELECTOR,
                    "div.vehica-car-card__info__single"
                )

                anio = infos[0].text if len(infos) > 0 else 0
                kilometraje = infos[1].text if len(infos) > 1 else 0
                combustible = infos[3].text if len(infos) > 3 else "N/A"

                # SEPARAR MARCA Y MODELO
                partes_nombre = nombre.split(" ", 1)

                marca = (
                    partes_nombre[0]
                    if len(partes_nombre) > 0
                    else "N/A"
                )

                modelo = (
                    partes_nombre[1]
                    if len(partes_nombre) > 1
                    else "N/A"
                )

                # GUARDAR DATOS
                lista_autos.append({

                    "marca": marca if marca else "N/A",

                    "modelo": modelo if modelo else "N/A",

                    "anio": anio if anio else 0,

                    "kilometraje": kilometraje if kilometraje else 0,

                    "combustible": combustible if combustible else "N/A",

                    "ciudad": "N/A",

                    "url": url_auto if url_auto else "N/A",

                    "precio": precio if precio else 0,

                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),

                    "grupo": NOMBRE_GRUPO,

                    "usuario": USUARIO

                })

            except Exception as e:

                continue

        print(f"Acumulado total: {len(lista_autos)} autos")

except Exception as e:

    print(f"Error Selenium: {e}")

finally:

    if driver is not None:

        try:
            driver.quit()
        except:
            pass

# --- GUARDAR EN MONGODB ---
try:

    client = MongoClient(
        "172.18.0.1",
        27017,
        serverSelectionTimeoutMS=5000
    )

    db = client["proyecto_bigdata"]

    coleccion = db["Valentini_scraping"]

    # LIMPIAR COLECCION
    coleccion.delete_many({})

    print("Coleccion limpiada.")

    if lista_autos:

        for d in lista_autos:

            # LIMPIAR PRECIO
            precio_limpio = (
                str(d["precio"])
                .replace(".", "")
                .replace("$", "")
                .replace(",", "")
                .strip()
            )

            try:
                d["precio"] = float(precio_limpio)
            except:
                d["precio"] = 0.0

            # LIMPIAR KILOMETRAJE
            km_limpio = (
                str(d["kilometraje"])
                .replace(".", "")
                .replace("KM", "")
                .replace("Km", "")
                .strip()
            )

            try:
                d["kilometraje"] = int(km_limpio)
            except:
                d["kilometraje"] = 0

            # LIMPIAR AÑO
            try:
                d["anio"] = int(d["anio"])
            except:
                d["anio"] = 0

        coleccion.insert_many(lista_autos)

        print(f"\n{len(lista_autos)} autos guardados en MongoDB.")

    else:

        print("No hay datos para guardar.")

except Exception as e:

    print(f"Error MongoDB: {e}")

Limpieza completada.
Navegador iniciado correctamente.

--- PROCESANDO PAGINA 1 ---
Autos encontrados: 12
Acumulado total: 12 autos

--- PROCESANDO PAGINA 2 ---
Autos encontrados: 12
Acumulado total: 24 autos

--- PROCESANDO PAGINA 3 ---
Autos encontrados: 0
Acumulado total: 24 autos

--- PROCESANDO PAGINA 4 ---
Autos encontrados: 0
Acumulado total: 24 autos

--- PROCESANDO PAGINA 5 ---
Autos encontrados: 0
Acumulado total: 24 autos
Coleccion limpiada.

24 autos guardados en MongoDB.
